# TCGA frozen-model expression-imputation benchmark

This notebook is the publication-facing record comparing our frozen 45.6M RNA-seq model with BulkFormer-50M and BulkFormer-147M on the same fixed 1,000 TCGA primary tumors. The shared-vocabulary analysis is primary; the native-vocabulary analysis is complementary. No model is fine-tuned.

## 1. Protocol

For each benchmark, 15%, 30%, and 50% of evaluable genes are masked under ten deterministic seeds. Metrics are computed only at masked positions. Shared-vocabulary masks are identical across all models for each sample, ratio, and seed. Gene-wise mean and median baselines use only values left unmasked under the corresponding condition. Results report the mean and standard deviation across masking seeds.

All checkpoints use their intended natural `log1p(TPM)` representation. Gene-length resources differ between model families; therefore MSE is native-target-space specific, while Pearson and Spearman provide the cleaner cross-model comparison.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_benchmark(start):
    for parent in (start.resolve(), *start.resolve().parents):
        if parent.name == 'tcga_imputation' and (parent / 'config.json').is_file():
            return parent
        candidate = parent / 'benchmarks/tcga_imputation'
        if (candidate / 'config.json').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the repository or one of its subdirectories.')

HERE = find_benchmark(Path.cwd())
WORK, RESULTS = HERE / 'work', HERE / 'results'
config = json.loads((HERE / 'config.json').read_text())
manifest = json.loads((RESULTS / 'preparation_manifest.json').read_text())
validation = json.loads((RESULTS / 'one_sample_validation.json').read_text())
samples = pd.read_parquet(RESULTS / 'selected_tcga_samples.parquet')
shared_genes = pd.read_csv(RESULTS / 'shared_genes.csv')
summary_path = RESULTS / 'summary_results.parquet'
per_seed_path = RESULTS / 'per_seed_results.parquet'
per_sample_path = RESULTS / 'per_sample_results.parquet'
print(f"Samples: {len(samples):,}; shared evaluable genes: {len(shared_genes):,}; validation: {validation['status']}")

Samples: 1,000; shared evaluable genes: 15,101; validation: passed


## 2. Fixed cohort and gene vocabularies

In [ ]:
display(pd.Series({
    'TCGA samples': manifest['samples'],
    'Selection population': manifest['selection_population'],
    'Our native genes': manifest['our_native_genes'],
    'BulkFormer native genes': manifest['bulkformer_native_genes'],
    'Model-vocabulary intersection': manifest['model_vocab_intersection'],
    'Shared evaluable genes': manifest['shared_evaluable_genes'],
}).to_frame('value'))
display(samples.cancer_type.value_counts().head(15).rename_axis('TCGA cancer type').to_frame('samples'))

,value
TCGA samples,1000
Selection population,TCGA Primary Tumor
Our native genes,15165
BulkFormer native genes,20010
Model-vocabulary intersection,15163
Shared evaluable genes,15101


,samples
TCGA cancer type,
Breast Invasive Carcinoma,118
Uterine Corpus Endometrial Carcinoma,63
Lung Adenocarcinoma,58
Lung Squamous Cell Carcinoma,56
Thyroid Carcinoma,55
Kidney Renal Clear Cell Carcinoma,54
Brain Lower Grade Glioma,53
Prostate Adenocarcinoma,48
Head and Neck Squamous Cell Carcinoma,46


## 3. One-sample end-to-end validation gate

In [ ]:
gate_rows = []
for benchmark, details in validation['benchmarks'].items():
    for model, values in details['models'].items():
        gate_rows.append({'benchmark': benchmark, 'model': model, **values})
gate = pd.DataFrame(gate_rows)
display(gate[['benchmark', 'model', 'input_shape', 'output_shape', 'masked_native_positions',
              'pearson', 'spearman', 'mse', 'finite_output']])

,benchmark,model,input_shape,output_shape,masked_native_positions,pearson,spearman,mse,finite_output
0,shared_vocab,ours_45.6m,"[1, 15165]","[1, 15165]",2265,0.952704,0.949775,0.282993,True
1,shared_vocab,bulkformer_50m,"[1, 20010]","[1, 20010]",2265,0.897482,0.890993,0.558075,True
2,shared_vocab,bulkformer_147m,"[1, 20010]","[1, 20010]",2265,0.925516,0.920093,0.409480,True
3,native_vocab,ours_45.6m,"[1, 15165]","[1, 15165]",2266,0.955572,0.952931,0.265647,True
4,native_vocab,bulkformer_50m,"[1, 20010]","[1, 20010]",2842,0.895076,0.884052,0.606815,True
5,native_vocab,bulkformer_147m,"[1, 20010]","[1, 20010]",2842,0.929382,0.921328,0.402904,True


## 4. Primary results: mean ± SD across ten masking seeds

In [ ]:
if summary_path.is_file():
    summary = pd.read_parquet(summary_path)
    expected_rows = 2 * len(config['mask_ratios']) * 7
    assert len(summary) == expected_rows, f'Incomplete summary: expected {expected_rows} rows, found {len(summary)}'
    assert summary.seeds.eq(len(config['mask_seeds'])).all(), 'One or more conditions lack all masking seeds'
    def pm(mean, sd, digits=3):
        return mean.map(lambda x: f'{x:.{digits}f}') + ' ± ' + sd.map(lambda x: f'{x:.{digits}f}')
    paper = summary.copy()
    paper['Pearson'] = pm(paper.pearson_mean, paper.pearson_sd)
    paper['Spearman'] = pm(paper.spearman_mean, paper.spearman_sd)
    paper['MSE'] = pm(paper.mse_mean, paper.mse_sd)
    display(paper[['benchmark', 'mask_ratio', 'method', 'target_space',
                   'evaluated_genes', 'Pearson', 'Spearman', 'MSE']])
else:
    print('Full benchmark results are not present yet. Run pipeline/run_imputation.py --run.')

Full benchmark results are not present yet. Run pipeline/run_imputation.py --run.


## 5. Performance across masking ratios

In [ ]:
if summary_path.is_file():
    primary = summary[summary.benchmark.eq('shared_vocab')]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    for (method, target_space), frame in primary.groupby(['method', 'target_space']):
        frame = frame.sort_values('mask_ratio')
        label = method if not method.startswith('gene_') else f'{method} ({target_space})'
        for ax, metric in zip(axes, ['pearson', 'spearman', 'mse']):
            ax.errorbar(frame.mask_ratio, frame[f'{metric}_mean'], yerr=frame[f'{metric}_sd'],
                        marker='o', capsize=2, label=label)
    for ax, title in zip(axes, ['Pearson ↑', 'Spearman ↑', 'MSE ↓']):
        ax.set(xlabel='Mask ratio', title=title)
    axes[0].set_ylabel('Mean across samples')
    axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1))
    fig.tight_layout()

## 6. Per-sample behavior

In [ ]:
if per_sample_path.is_file():
    per_sample = pd.read_parquet(per_sample_path)
    display(per_sample.groupby(['benchmark', 'method', 'mask_ratio'])
            .agg(samples=('sample_id', 'nunique'), median_pearson=('pearson', 'median'),
                 q10_pearson=('pearson', lambda x: x.quantile(.10)),
                 median_mse=('mse', 'median')).reset_index())

## 7. Interpretation checklist

The paper interpretation should prioritize `shared_vocab`, compare frozen models at every masking ratio, and report variability across masking seeds. The native-vocabulary result answers a different question—performance under each model's intended gene space—and should not replace the controlled shared-gene comparison. MSE comparisons must acknowledge native log-TPM target spaces. Conclusions should distinguish reconstruction of held-out expression values from downstream biological utility.